## Imports

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import sys; sys.path.insert(0, '..')
from datatypes.jd_data_type import JobDescription
from utils.jd_util import JobDescriptionUtil
from datatypes.resume_data_type import Resume
from utils.resume_util import ResumeUtil
from utils.scorer import Scorer
import pandas as pd
from tqdm import tqdm

## Sample Tests

In [17]:
sample_resume = '{"description":"Results-driven Software Developer with 3+ years of experience in Spring boot, React JS, and SQL, specializing in performance optimization and cybersecurity. Skilled in solving complex problems and leading projects to success. Looking to leverage my technical expertise to drive innovation and deliver impactful software solutions in a dynamic environment.","education":[{"school_name":"Karunya Institute of Technology and Science","year":"2017","degree":"Bachelor of Technology in Computer Science"}],"skills":["Java","JavaScript","SQL","HTML","CSS","SpringBoot","React.js","Node.js","SQL"],"experience":[{"company_name":"Zoho Corp Pvt Ltd.","start_year":"2021","end_year":"Present","descriptions":["Introduced URL Monitoring for OpManagement MSP Edition to monitor the response time, availability, and reachability of websites managed by enterprises using HTTPClient jar, improving overall monitoring eciency.","Developed Hardware Monitoring for 25 major vendors, enabling the monitoring of hardware sensors such as fan speed, temperature, and power supplies in network devices like switches, routers, and rewalls using SNMP, IPMI, and WMI protocols. This resulted in proactive alerts for customers if any device malfunctioned or exceeded thresholds.","Optimized the performance of the URL Monitors page by loading data through a single SQL query, rather than retrieving the uptime for each URL separately, resulting in a signicant reduction in load times by 95%.","Implemented alerts for memory and CPU utilization of customer devices, including the top three high utilization processes and detailed memory and CPU usage for all active processes, enhancing system visibility and performance monitoring.","Passionate about Cybersecurity, actively identied and xed security vulnerabilities, including SQL Injections, XSS, and RCEs, through rigorous testing and analysis of our product. This proactive approach enhanced the products security posture and reduced potential security risks by 30%.","Enhanced the product to use one session only while collecting the data from IPMI RestAPIs to monitor Hardwares of the devices instead of initiating more number of sessions each time. This reduced the number of load to the devices by 40%.","Optimized IPMI monitoring performance by maintaining a single session throughout data collection, rather than creating multiple sessions for each request. This enhancement reduced the load on monitored devices by 80%, improving eciency and system stability.","Resolved customer issues promptly by addressing their requirements and xing raised issues, improving customer satisfaction and response times by 60%."]},{"company_name":null,"start_year":null,"end_year":null,"descriptions":[]},{"company_name":null,"start_year":null,"end_year":null,"descriptions":[]}],"projects":[],"certifications":[]}'

sample_jd = '{"job_summary":"Build the user-facing layer of AI-powered workflows used by real customers. Define fast, intuitive interfaces in a product with meaningful workflow complexity. Shape a design system and frontend foundation that scales with the company.","role_description":["Collaborate with design and backend teams to ship seamless experiences in React from Figma designs","Build and maintain reusable UI components and systems that scale across customers","Implement frontend features that interact with AI services such as document review, entity extraction, and quote comparison","Rapidly prototype and iterate based on user feedback and evolving priorities"],"required_skills":["2+ years building frontend applications with React","Proficiency with design tools and handoff workflows, especially Figma.","Strong product sense and the ability to translate complex workflows into clean UX.","Familiarity with REST or GraphQL APIs and state management patterns.","Good judgment about where polish matters and where speed matters"],"general_skills":["Experience with data-rich or document-heavy interfaces","Experience building LLM-integrated product experiences","Modern dev-tool fluency (e.g., Cursor)","Experience building LLM-integrated product experiences"],"soft_skills":[],"min_experience_yrs":"2","max_experience_yrs":"","degree_needed":"Bachelor\'s or Master\'s degree in a technical discipline","certifications_required":[]}'

sample_score = 63.5

In [19]:
resume: Resume = ResumeUtil.create(sample_resume)
job_description: JobDescription = JobDescriptionUtil.create(sample_jd)

In [20]:
ResumeUtil.format(resume)

{
    "description": "Results-driven Software Developer with 3+ years of experience in Spring boot, React JS, and SQL, specializing in performance optimization and cybersecurity. Skilled in solving complex problems and leading projects to success. Looking to leverage my technical expertise to drive innovation and deliver impactful software solutions in a dynamic environment.",
    "education": [
        {
            "school_name": "Karunya Institute of Technology and Science",
            "year": "2017",
            "degree": "Bachelor of Technology in Computer Science"
        }
    ],
    "skills": [
        "Java",
        "JavaScript",
        "SQL",
        "HTML",
        "CSS",
        "SpringBoot",
        "React.js",
        "Node.js",
        "SQL"
    ],
    "experience": [
        {
            "company_name": "Zoho Corp Pvt Ltd.",
            "start_year": "2021",
            "end_year": "Present",
            "descriptions": [
                "Introduced URL Monitoring f

In [21]:
JobDescriptionUtil.format(job_description)

{
    "job_summary": "Build the user-facing layer of AI-powered workflows used by real customers. Define fast, intuitive interfaces in a product with meaningful workflow complexity. Shape a design system and frontend foundation that scales with the company.",
    "role_description": [
        "Collaborate with design and backend teams to ship seamless experiences in React from Figma designs",
        "Build and maintain reusable UI components and systems that scale across customers",
        "Implement frontend features that interact with AI services such as document review, entity extraction, and quote comparison",
        "Rapidly prototype and iterate based on user feedback and evolving priorities"
    ],
    "required_skills": [
        "2+ years building frontend applications with React",
        "Proficiency with design tools and handoff workflows, especially Figma.",
        "Strong product sense and the ability to translate complex workflows into clean UX.",
        "Familiarit

## Constants

In [22]:
model_name = "Qwen/Qwen3-Embedding-0.6B"

import torch
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
device

'mps'

## Model

In [23]:
model = SentenceTransformer(model_name, device=device)
model

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'Qwen3Model'})
  (1): Pooling({'embedding_dimension': 1024, 'pooling_mode': 'lasttoken', 'include_prompt': True})
  (2): Normalize({})
)

In [24]:
# Sample example...
a = ['Strong experience with Node.js and Nest.js (3–6 years)']
b = ['Node JS']
a_embs = np.asarray(model.encode(a))
b_embs = np.asarray(model.encode(b))

cosine_similarity(np.atleast_2d(a_embs), np.atleast_2d(b_embs))

array([[0.6951053]], dtype=float32)

## Scoring

In [25]:
scorer = Scorer(model_name)

Scorer :: Inititalizing the model


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Scorer :: Model Qwen/Qwen3-Embedding-0.6B is initiated on device 'mps'


In [26]:
scorer.get_overall_score(resume, job_description)

np.float32(72.4709)

## Testing

In [27]:
df = pd.read_csv("../datasets/resume_jd_test_dataset.csv")
df.head()

,resume,job_description,summary_match_score,role_match_score,required_skill_score,general_skill_score,soft_skill_score,experience_years_match_score,education_match_score,certification_match_score,overall_score
0,"{""description"": ""Backend Developer with 2 year...","{""job_summary"": ""We are hiring a Backend Engin...",1.67,16.68,11.25,2.33,1.50,14.67,8.0,12.0,68.10
1,"{""description"": ""Results-driven DevOps Enginee...","{""job_summary"": ""We are hiring a DevOps Engine...",0.83,11.07,11.29,3.50,0.00,22.00,8.0,12.0,68.69
2,"{""description"": ""Passionate Software Engineer ...","{""job_summary"": ""We are seeking a QA Automatio...",0.00,2.13,4.60,1.75,1.50,22.00,8.0,12.0,51.98
3,"{""description"": ""Results-driven AI/ML Engineer...","{""job_summary"": ""We are hiring an AI/ML Engine...",1.67,8.52,10.46,3.50,0.75,8.80,8.0,12.0,53.70
4,"{""description"": ""Results-driven Software Engin...","{""job_summary"": ""We are seeking a QA Automatio...",0.00,3.41,3.44,1.17,0.00,22.00,8.0,12.0,50.02


In [28]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 11 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   resume                        500 non-null    str    
 1   job_description               500 non-null    str    
 2   summary_match_score           500 non-null    float64
 3   role_match_score              500 non-null    float64
 4   required_skill_score          500 non-null    float64
 5   general_skill_score           500 non-null    float64
 6   soft_skill_score              500 non-null    float64
 7   experience_years_match_score  500 non-null    float64
 8   education_match_score         500 non-null    float64
 9   certification_match_score     500 non-null    float64
 10  overall_score                 500 non-null    float64
dtypes: float64(9), str(2)
memory usage: 1.8 MB


In [29]:
def get_average_score_diff(scorer, test_df):
    length = len(test_df)

    summary_match_scores = []
    role_match_scores = []
    required_skill_scores = []
    general_skill_scores = []
    soft_skill_scores = []
    experience_years_match_scores = []
    education_match_scores = []
    certification_match_scores = []
    overall_scores = []
    for index in tqdm(range(length)):
        row = df.iloc[index]
        resume = ResumeUtil.create(row["resume"])
        job_description = JobDescriptionUtil.create(row["job_description"])

        summary_match_score = scorer.get_summary_score(resume, job_description)
        role_match_score = scorer.get_role_score(resume, job_description)
        required_skill_score = scorer.get_skills_score(resume, job_description, "required")
        general_skill_score = scorer.get_skills_score(resume, job_description, "general")
        soft_skill_score = scorer.get_skills_score(resume, job_description, "soft")
        experience_years_match_score = scorer.get_experience_score(resume, job_description)
        education_match_score = scorer.get_education_score(resume, job_description)
        certification_match_score = scorer.get_certifications_score(resume, job_description)
        overall_score = summary_match_score + role_match_score + required_skill_score + general_skill_score + soft_skill_score + experience_years_match_score + education_match_score + certification_match_score

        summary_match_scores.append(summary_match_score)
        role_match_scores.append(role_match_score)
        required_skill_scores.append(required_skill_score)
        general_skill_scores.append(general_skill_score)
        soft_skill_scores.append(soft_skill_score)
        experience_years_match_scores.append(experience_years_match_score)
        education_match_scores.append(education_match_score)
        certification_match_scores.append(certification_match_score)
        overall_scores.append(overall_score)

    average_test_score = {
        "summary_match_score": np.sum(np.asarray(np.abs(test_df["summary_match_score"] - np.array(summary_match_scores))))/length,
        "role_match_score": np.sum(np.asarray(np.abs(test_df["role_match_score"] - np.array(role_match_scores))))/length,
        "required_skill_score": np.sum(np.asarray(np.abs(test_df["required_skill_score"] - np.array(required_skill_scores))))/length,
        "general_skill_score": np.sum(np.asarray(np.abs(test_df["general_skill_score"] - np.array(general_skill_scores))))/length,
        "soft_skill_score": np.sum(np.asarray(np.abs(test_df["soft_skill_score"] - np.array(soft_skill_scores))))/length,
        "experience_years_match_score": np.sum(np.asarray(np.abs(test_df["experience_years_match_score"] - np.array(experience_years_match_scores))))/length,
        "education_match_score": np.sum(np.asarray(np.abs(test_df["education_match_score"] - np.array(education_match_scores))))/length,
        "certification_match_score": np.sum(np.asarray(np.abs(test_df["certification_match_score"] - np.array(certification_match_scores))))/length,
        "overall_score": np.sum(np.asarray(np.abs(test_df["overall_score"] - np.array(overall_scores))))/length,
    }

    return average_test_score

In [30]:
scorer = Scorer(model_name)

Scorer :: Inititalizing the model


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Scorer :: Model Qwen/Qwen3-Embedding-0.6B is initiated on device 'mps'


In [31]:
scorer.get_parameters()

{'certifications_penalty': 0.85,
 'projects_pentalty': 0.8,
 'skill_match_threshold': 0.55,
 'education_match_threshold': 0.5705,
 'certifications_match_threshold': 0.605}

In [32]:
get_average_score_diff(scorer, df)

100%|██████████| 500/500 [05:13<00:00,  1.59it/s]


{'summary_match_score': np.float64(1.7034584572982787),
 'role_match_score': np.float64(3.251927129440308),
 'required_skill_score': np.float64(8.698355555555557),
 'general_skill_score': np.float64(3.863813333333333),
 'soft_skill_score': np.float64(0.7245),
 'experience_years_match_score': np.float64(2.7840418181818176),
 'education_match_score': np.float64(0.744),
 'certification_match_score': np.float64(0.84),
 'overall_score': np.float64(16.818762495422366)}